# Varying δ experiments for Private SpiderBoost

Sweep over the privacy parameter δ = 1/n^α for α ∈ {1.1, 1.4, 1.7, 2.0} at fixed ε = 1.0.
Results are compared against a non-private SPIDER baseline (SGD with variance-reduced gradient estimates, same architecture, matched gradient budget).

## 1. Imports

In [ ]:
import sys
from pathlib import Path

HERE = Path('.').resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import jax
import jax.numpy as jnp
import numpy as np
from sklearn.metrics import roc_auc_score

import dimma
from dimma import TrainConfig, compute_noise_scales
from dimma.datasets import load_criteo

import model
import visualization as viz

## 2. Data and sweep configuration

In [ ]:
FIGS_DIR = HERE / 'figs'
FIGS_DIR.mkdir(exist_ok=True)

data = load_criteo(features='integer', test_fraction=0.2, seed=0, device='cpu')
x_train, y_train = data.x_train, data.y_train
x_test, y_test = data.x_test, data.y_test
n_train, d = x_train.shape

# Sweep configuration — extend SEEDS to average over multiple runs.
EXPONENTS = [1.1, 1.4, 1.7, 2.0]   # δ = 1 / n_train ** α
SEEDS = [0]
EPSILON = 1.0
HIDDEN_DIMS = (64, 32)

BASE = dict(
    epsilon=EPSILON,
    L0=3.0,
    L1=5.0,
    T=200,
    q=30,
    b1=8192,
    b2=512,
    eta=0.01,
)

print(f'n_train={n_train}, d={d}')
print(f'Delta values: {[f"1/n^{a} = {1/n_train**a:.2e}" for a in EXPONENTS]}')

## 3. AUC helper

In [ ]:
@jax.jit
def predict_logits(params, x):
    return model.forward(params, x)

def evaluate_auc(params, x, y):
    logits = predict_logits(params, x)
    return float(roc_auc_score(np.asarray(y), np.asarray(logits)))

## 4. Non-private baseline

Train a non-private MLP using `model.train_spider` — the same anchor/variation step
structure as Private SpiderBoost, but without gradient clipping or noise.
Parameters are updated via SGD: `w_{t+1} = w_t - η · grad_estimate_t`, where
`grad_estimate_t` is the variance-reduced SPIDER estimate (full-batch mean on anchor
steps, incremental correction on variation steps).
This controls for the optimizer design; the only difference from the DP runs is
the absence of privacy mechanisms.

In [ ]:
print('Training non-private baseline (SPIDER, no DP)...')
auc_baseline_per_seed = []
for seed in SEEDS:
    key = jax.random.PRNGKey(seed)
    init_p = model.init_params(key, input_dim=d, hidden_dims=HIDDEN_DIMS)
    res_np = model.train_spider(
        x_train, y_train,
        init_params=init_p,
        T=BASE['T'],
        q=BASE['q'],
        b1=BASE['b1'],
        b2=BASE['b2'],
        eta=BASE['eta'],
        seed=seed,
    )
    auc_fin = evaluate_auc(res_np.params_final, x_test, y_test)
    auc_rand = evaluate_auc(res_np.params_random, x_test, y_test)
    auc_baseline_per_seed.append(auc_fin)
    print(f'  seed={seed}  AUC(w_T)={auc_fin:.4f}  AUC(w\u0305)={auc_rand:.4f}  ({sum(res_np.history.wall_time_s):.1f}s)')

auc_baseline = float(np.mean(auc_baseline_per_seed))
print(f'Non-private baseline AUC (mean over {len(SEEDS)} seed(s)): {auc_baseline:.4f}')

## 5. δ sweep

In [ ]:
# results[α] = list of (auc_random, auc_final) tuples, one per seed.
results = {alpha: [] for alpha in EXPONENTS}

for alpha in EXPONENTS:
    delta = 1.0 / (n_train ** alpha)
    print(f'--- α={alpha}  δ={delta:.2e} ---')
    for seed in SEEDS:
        cfg = TrainConfig(delta=delta, seed=seed, **BASE)
        noise_scales = compute_noise_scales(
            L0=cfg.L0, L1=cfg.L1, epsilon=cfg.epsilon, delta=cfg.delta,
            T=cfg.T, q=cfg.q, n=n_train, b1=cfg.b1, b2=cfg.b2,
        )
        key = jax.random.PRNGKey(seed)
        init_params = model.init_params(key, input_dim=d, hidden_dims=HIDDEN_DIMS)
        res = dimma.train(
            x_train, y_train,
            per_sample_loss_fn=model.per_sample_bce_loss,
            init_params=init_params,
            config=cfg,
            noise_scales=noise_scales,
            sampler='poisson',
        )
        a_rand = evaluate_auc(res.params_random, x_test, y_test)
        a_fin = evaluate_auc(res.params_final, x_test, y_test)
        results[alpha].append((a_rand, a_fin))
        print(f'  seed={seed}  AUC(w\u0305)={a_rand:.4f}  AUC(w_T)={a_fin:.4f}  ({sum(res.history.wall_time_s):.1f}s)')

# Average over seeds.
auc_random_list = [float(np.mean([r[0] for r in results[a]])) for a in EXPONENTS]
auc_final_list  = [float(np.mean([r[1] for r in results[a]])) for a in EXPONENTS]

print('\nSummary (mean over seeds):')
for alpha, a_rand, a_fin in zip(EXPONENTS, auc_random_list, auc_final_list):
    print(f'  α={alpha}  AUC(w\u0305)={a_rand:.4f}  AUC(w_T)={a_fin:.4f}')

## 6. Plot

In [ ]:
_ = viz.plot_delta_sweep(
    EXPONENTS, auc_random_list, auc_final_list, auc_baseline,
    FIGS_DIR / 'varying_delta_experiments.png',
)
print({
    'exponents': EXPONENTS,
    'auc_random': auc_random_list,
    'auc_final': auc_final_list,
    'auc_baseline': auc_baseline,
})